# AgentRegistry, end to end: scaffold a dice agent, add approved MCP tools, deploy to kagent **and** AWS Bedrock AgentCore, then govern the tools

A developer builds an agent with `arctl`, runs it locally, pulls in **approved MCP tool servers** from the registry catalog, publishes the agent, and deploys the *same* published agent to two runtimes (Solo Enterprise for kagent on kind, and AWS Bedrock AgentCore). Finally a platform owner restricts which MCP tools the agent may call with an **AccessPolicy**, enforced at an agentgateway waypoint.

> **Kernel:** pick **Bash** (top-right). One-time engineer setup lives in `scripts/` (see `README.md`).

## Open the consoles

Run this once **in a terminal** — it port-forwards the kagent UI and opens both tabs (no `/etc/hosts` edit needed):

```sh
./scripts/open-consoles.sh
```

| Console | URL | Login |
|---|---|---|
| AgentRegistry UI | http://localhost:12121 | — |
| kagent UI | http://localhost:18007 | alice / alice |
| AWS Bedrock AgentCore | the AWS console (manual) | — |

## Connect to the platform

Loads creds, puts `arctl` on the path, mints a catalog token.

In [ ]:
[ -d agentregistry-agentcore-kind ] && cd agentregistry-agentcore-kind || :; source scripts/connect.sh && rm -rf agentdemo

## 1. Create a new agent project

Scaffolds the agent into `agentdemo/`. Open it in the Explorer. The generated agent has two **local** tools: `roll_die` and `check_prime`.

In [ ]:
arctl init agent agentdemo --framework adk --language python --model-provider anthropic --model-name claude-haiku-4-5

In [ ]:
cat agentdemo/agentdemo/agent.py

## 2. Build and run it locally

Build the image, then run it in an interactive chat **in a terminal** (`arctl run` is interactive, so not a notebook cell):

```sh
arctl run ./agentdemo
```
Ask: *"Roll a 20-sided die and tell me whether the result is a prime number."* — it uses the local `roll_die` + `check_prime` tools.

In [ ]:
arctl build ./agentdemo

## 3. Browse the approved tool-server catalog

Rather than wiring arbitrary MCP servers, developers pull from an **approved catalog**. Open the **AgentRegistry UI** (http://localhost:12121) → **Tool Servers** to see `everything-server` and `my-mcp`. The same list from the CLI:

In [ ]:
arctl get mcpservers

## 4. Add the approved `everything-server` to the agent

`arctl` is declarative — there is no `add-mcp` verb. Adding an approved MCP server from the catalog means referencing it in the agent's `agent.yaml` under `spec.mcpServers`; the registry resolves that into a tool server beside the agent at deploy time. (`arctl init agent --mcp <name@tag>` wires it at scaffold time — we scaffolded first, so we add it now.)

In [ ]:
# Append the approved server to spec.mcpServers (cell below shows the result):
yq -i '.spec.mcpServers += [{"kind":"MCPServer","name":"demo/everything-server","tag":"latest"}]' agentdemo/agent.yaml

In [ ]:
yq '.spec.mcpServers' agentdemo/agent.yaml

Now add the approved **dice-game** skill — it gives the agent a house format for reporting rolls and prime checks. A skill is behavioural guidance, not a tool, and the Agent kind has no `skills` field, so its body folds into the agent's system instruction (`build_instruction(...)` in `agent.py`).

In [ ]:
# Show the approved skill from the catalog, then fold its body (everything after
# the front-matter) into the agent's system instruction in agent.py:
cat skill/dice-game/SKILL.md

python3 - <<'EOF'
import re
body = re.sub(r'^---\n.*?\n---\n', '', open("skill/dice-game/SKILL.md").read(), count=1, flags=re.S).strip()
p = "agentdemo/agentdemo/agent.py"
src = open(p).read()
src = re.sub(r'build_instruction\(\s*"""(.*?)"""\s*\)',
             'build_instruction("""\n' + body.replace("\\", "\\\\") + '\n""")', src, count=1, flags=re.S)
open(p, "w").write(src)
print("\nbaked dice-game skill into", p)
EOF

## 5. Run again — now the agent has the MCP tools

Run the MCP server and the agent locally, in **two terminals**:

```sh
# terminal 1 — the approved tool server
arctl run ./mcp/everything-server

# terminal 2 — the agent (reaches the server on host.docker.internal:3000)
arctl run ./agentdemo
```
- Ask *"What tools do you have access to?"* — alongside the local `roll_die`/`check_prime` you now see the MCP tools (`sum`, `echo`, `to_uppercase`, `reverse_text`, ...).
- Ask *"Roll a 13-sided die, add 5 to the result, then tell me if it's prime."* — the chain calls the MCP **`sum`** tool. (The MCP server runs as its own sandboxed container; the agent calls out to it over HTTP — the tools are not copied into the agent.)

## 6. Build (multi-arch), publish, and push the source

Multi-arch so the same image runs on kagent (arm64 here) and AgentCore (amd64).

In [ ]:
arctl build ./agentdemo --platform linux/amd64,linux/arm64 --push

In [ ]:
arctl apply -f agentdemo/agent.yaml

AgentCore clones the agent **source** from git at deploy time, so push it to your agent repo (the engineer set `AGENT_GIT_URL` in `.env.local`):

In [ ]:
./scripts/push-source.sh

## 7. Deploy onto kagent (runtime #1)

Binds the agent to the `kind-kagent` runtime. The registry stands up the agent **and** the referenced MCP server as pods, wired together.

In [ ]:
envsubst < yaml/deploy-kagent.yaml | arctl apply -f -

In [ ]:
kubectl --context kind-$CLUSTER_NAME -n kagent get pods | grep -E 'agentdemo|everything-server|my-mcp'

## 8. Deploy the same agent to AWS Bedrock AgentCore (runtime #2)

Same published agent, different `runtimeRef`. On AgentCore it uses Bedrock Claude via the AWS role (no API key). Requires an AWS account; skip for a local-only run.

In [ ]:
source scripts/aws-login.sh

In [ ]:
./scripts/agentcore-deploy.sh

In [ ]:
./scripts/ac-invoke.sh "Roll a 13-sided die, add 5 to the result, then tell me if it is prime."

## 9. Run it from the kagent UI — traces and spans

Open the **kagent UI** (http://localhost:18007), invoke `agentdemo`, and watch the OTEL trace: the agent calls `roll_die`, then the MCP `sum` tool, then `check_prime`, span by span.

## 10. Govern the tools with an AccessPolicy (least privilege)

The `everything-server` exposes a sensitive `printenv` tool. A platform owner restricts the agent to only the tools it needs. `accesspolicy-on.sh` labels the MCP server for an **agentgateway waypoint** and applies a kagent **AccessPolicy** that denies `printenv`.

In [ ]:
./scripts/accesspolicy-on.sh

The policy is a declarative custom resource — show it on the CLI:

In [ ]:
kubectl --context kind-$CLUSTER_NAME -n kagent get accesspolicy deny-printenv -o yaml

Ask the agent for its tools again (or re-ask in the kagent UI) — `printenv` is gone; `sum` and the rest still work:

In [ ]:
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

## Reset / teardown

```sh
./scripts/accesspolicy-off.sh   # restore the full tool list
./scripts/reset.sh              # back to start (clears agentdemo, deployments)
./scripts/cleanup.sh            # full teardown (cluster, daemon, registry, AWS)
```